# NadiKampus — Pelabelan Sentimen Telegram (Gold Set)

Notebook ini membuat **label sentimen (positif / netral / negatif)** untuk post Telegram (menfess). Pasangan dari notebook pelabelan GForm: format label, UI, dan penyimpanan dibuat sama supaya hasilnya bisa digabung di bagian 8 notebook GForm.

**Beda dengan GForm:** korpus Telegram besar (ribuan post), jadi **tidak semuanya dilabeli**. Kita ambil **sampel acak 300 post** sebagai *gold set* (target Pedoman: Telegram sekitar 200-300). Sampel ini dipakai untuk melatih dan menguji metode sentimen, lalu modelnya diterapkan ke seluruh korpus.

**Alur:** `muat korpus → ambil sampel acak → beri label (UI / Excel) → cek sebaran → hitung kesepakatan antar-pelabel → bahas selisih → ekspor`

### Panduan label (pakai aturan yang sama untuk semua pelabel)

Nilai **sikap penulis** pada isi post, bukan topiknya. Post tentang "tugas" bisa positif, netral, atau negatif tergantung nadanya.

| Label | Ciri | Contoh |
|---|---|---|
| **Positif** | apresiasi, terima kasih, semangat, kepuasan, kabar baik | "Makasih banyak kating yang udah bantu, lancar banget urusannya!" |
| **Netral** | pertanyaan, permintaan info, jual-beli, pengumuman, kehilangan/penemuan barang, fakta datar | "Ada yang tau jadwal KRS gelombang 2?", "Dijual laptop Asus, nego" |
| **Negatif** | keluhan, kekesalan, kritik, kecemasan, curhat sedih | "Dosennya ngilang terus, nilai gak keluar udah sebulan" |

Aturan tambahan:

* **Kalimat campuran dengan kata pertentangan (tetapi / namun / sayangnya / tapi)** → nilai **maksud utamanya**, biasanya di *belakang* kata pertentangan. Setelahnya keluhan → **Negatif**; setelahnya hanya harapan atau saran → **Netral**.
* **Negasi membalik makna**: "tidak ramah", "kurang jelas", "belum membantu" → **Negatif**.
* **Pertanyaan bernada keluhan** ("kok layanan kampus lambat banget ya?") → **Negatif**. Pertanyaan murni minta info ("jam buka perpus?") → **Netral**.
* **Sarkasme / slang / bahasa Jawa / campur Inggris**: nilai niat sebenarnya. Emoji dan huruf besar ("MANTAP", "😭") sering menunjukkan nada; gunakan sebagai petunjuk, bukan aturan mutlak.
* **Netral hanya untuk teks tanpa sikap.** Jangan memakai Netral sebagai tempat menaruh yang sulit diputuskan.
* **Jual-beli, lost & found, cari teman/info** → **Netral**, kecuali ada nada jelas (misal "tolong banget, laptop saya hilang 😭" bernada cemas → Negatif).
* Ragu → pilih label terbaik lalu **centang "Tandai ragu"** (dibahas bersama). Jangan menebak berdasarkan identitas atau prodi.

## 1. Setup & konfigurasi

In [1]:
# Jalankan sekali kalau belum terpasang, lalu restart kernel:
# %pip install pandas pyarrow ipywidgets openpyxl scikit-learn

In [2]:
import os, html
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

In [11]:
# ====================== KONFIGURASI ======================
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir()), Path.cwd())
CORPUS_PATH = next((p for p in [ROOT / "data" / "processed" / "nadikampus_corpus_all.parquet",
                                ROOT / "data" / "nadikampus_corpus_all.parquet",
                                ROOT / "nadikampus_corpus_all.parquet",
                                Path.cwd() / "nadikampus_corpus_all.parquet"] if p.exists()),
                   ROOT / "data" / "processed" / "nadikampus_corpus_all.parquet")
OUT_DIR = ROOT / "data" / "labeling"            # jangan di-commit ke GitHub (berisi teks post)

ANNOTATOR = "annotator1"   # <- isi namamu. Tiap pelabel punya file sendiri: labels_telegram_<nama>.parquet
SEED = 42                  # semua pelabel WAJIB memakai SEED dan pengaturan sampel yang sama
N_SAMPLE = 500             # ukuran gold set Telegram. Boleh dinaikkan nanti: sampel lama tetap ada di depan
ONLY_VALID = True          # hanya post is_valid
MIN_WORDS = 3              # buang post terlalu pendek (mis. "up", "hai"); 0 = tidak dibuang
DEDUP = True               # buang post dengan isi sama persis (repost jual-beli dll.) supaya tidak bocor ke test

SENT_LABELS  = ["negative", "neutral", "positive"]
SENT_DISPLAY = {"negative": "😠 Negatif", "neutral": "😐 Netral", "positive": "🙂 Positif"}

## 2. Muat korpus Telegram & ambil sampel acak

Korpus diacak dengan `SEED` lalu diambil `N_SAMPLE` pertama. Karena urutannya tetap, **menaikkan `N_SAMPLE` hanya menambah di belakang** dan 100-150 post pertama selalu sama untuk semua pelabel (dipakai untuk menghitung kesepakatan).

In [12]:
assert CORPUS_PATH.exists(), f"File korpus tidak ditemukan: {CORPUS_PATH.resolve()} -> ubah CORPUS_PATH"
corpus = pd.read_parquet(CORPUS_PATH)

tg_all = corpus[corpus["source"].astype(str) == "telegram"].copy()
n_all = len(tg_all)
if ONLY_VALID and "is_valid" in tg_all:
    tg_all = tg_all[tg_all["is_valid"]]
if MIN_WORDS and "n_words" in tg_all:
    tg_all = tg_all[tg_all["n_words"] >= MIN_WORDS]
if DEDUP:
    tg_all = tg_all.drop_duplicates("text_clean")

tg_all = tg_all.sort_values("doc_id").sample(frac=1, random_state=SEED).reset_index(drop=True)   # acak, tetap sama antar pelabel
tg = tg_all.head(N_SAMPLE).reset_index(drop=True)

print(f"Telegram di korpus: {n_all} | lolos filter: {len(tg_all)} | sampel untuk dilabeli: {len(tg)}")
if "n_words" in tg:
    print("Panjang post (kata):", tg["n_words"].describe()[["min", "50%", "75%", "max"]].round(1).to_dict())
if "doc_type" in tg:
    print("Jenis dokumen di sampel:", tg["doc_type"].astype(str).value_counts().to_dict())
tg[["doc_id", "text_raw"]].head(5)

Telegram di korpus: 4807 | lolos filter: 4789 | sampel untuk dilabeli: 500
Panjang post (kata): {'min': 4.0, '50%': 15.0, '75%': 24.0, 'max': 49.0}
Jenis dokumen di sampel: {'post': 500}


,doc_id,text_raw
0,tg_10572,"Sedih? Iya tapi juga bangga, sahabatku kini melangkah ke jalan yang baru, menapak kaki nya sebagai seorang kepala ke..."
1,tg_10912,Kalo ada barang hilang di teater itu kemungkinan masih disana apa dibawa kemana ges
2,tg_11720,Ke kampus makin mual lihat perlakuan munafik yang katanya jujur malah absen tapi nda Dateng kelas 🤢\r\nKatanya ningk...
3,tg_10718,"Hey chel, ummm. I dreamt about u last night, and i just wanna let u know that... i miss u"
4,tg_11910,info nasi bento 10k yg jual pagi pagi daerah keputih


## 3. Penyimpanan label

Setiap klik langsung tersimpan ke `data/labeling/labels_telegram_<ANNOTATOR>.parquet` (ditulis aman: file sementara lalu diganti). Menutup dan membuka lagi notebook **tidak menghilangkan label**; antrean hanya berisi post yang belum dilabeli.

In [13]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
STORE_PATH = OUT_DIR / f"labels_telegram_{ANNOTATOR}.parquet"
STORE_COLS = ["doc_id", "label_sentiment", "ragu", "labeled_at", "annotator"]

def load_store():
    if STORE_PATH.exists():
        return pd.read_parquet(STORE_PATH)
    return pd.DataFrame({c: pd.Series(dtype="object") for c in STORE_COLS})

def save_store(store):
    tmp = STORE_PATH.with_suffix(".tmp")
    store.to_parquet(tmp, index=False)
    os.replace(tmp, STORE_PATH)

def upsert(store, doc_id, sentiment, ragu=False):
    row = pd.DataFrame([{"doc_id": doc_id, "label_sentiment": sentiment, "ragu": bool(ragu),
                         "labeled_at": datetime.now().isoformat(timespec="seconds"), "annotator": ANNOTATOR}])
    store = store[store["doc_id"] != doc_id]
    return pd.concat([store, row], ignore_index=True) if len(store) else row

store = load_store()
print(f"Label tersimpan untuk '{ANNOTATOR}': {len(store)} dari {len(tg)} post")

class LabelSession:
    # Logika antrean labeling (dipisah dari UI supaya mudah diuji)
    def __init__(self, rows, store):
        self.rows = rows.set_index("doc_id", drop=False)
        self.store = store
        done = set(store["doc_id"])
        self.queue = [d for d in rows["doc_id"] if d not in done]
        self.history = []

    def current(self):
        return self.queue[0] if self.queue else None

    def existing(self, doc_id):
        r = self.store[self.store["doc_id"] == doc_id]
        return (r["label_sentiment"].iloc[0], bool(r["ragu"].iloc[0])) if len(r) else (None, False)

    def label(self, sentiment, ragu=False):
        doc = self.queue.pop(0)
        self.store = upsert(self.store, doc, sentiment, ragu)
        save_store(self.store)
        self.history.append(doc)

    def skip(self):
        if len(self.queue) > 1:
            self.queue.append(self.queue.pop(0))

    def back(self):
        if self.history:
            self.queue.insert(0, self.history.pop())

    def done_count(self):
        return int(self.rows.index.isin(self.store["doc_id"]).sum())

    def total(self):
        return len(self.rows)

Label tersimpan untuk 'annotator1': 300 dari 500 post


## 4. Labeling di notebook (UI interaktif)

Jalankan sel di bawah, lalu **satu klik pada tombol sentimen langsung menyimpan dan pindah** ke post berikutnya.

* **Tandai ragu** dicentang *sebelum* mengklik tombol sentimen, untuk kasus ambigu yang perlu dibahas.
* **Kembali** membuka lagi post yang baru dilabeli (label lama terisi; klik tombol lain untuk mengubah).
* **Lewati** memindahkan post ke akhir antrean.

Kalau widget tidak tampil, pakai jalur Excel di bagian 5.

In [14]:
try:
    import ipywidgets as W
    from IPython.display import display
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets belum terpasang -> jalankan:  %pip install ipywidgets   (atau pakai jalur Excel di bagian 5)")

def _esc(t):
    return html.escape(str(t)).replace("\n", "<br>")

HINT = ("<div style='font-size:12px;opacity:.7;margin:6px 0'>"
        "😠 keluhan / kesal / kritik / cemas &nbsp;·&nbsp; 😐 tanya / info / jual-beli / pengumuman "
        "&nbsp;·&nbsp; 🙂 apresiasi / semangat / puas</div>")

class LabelUI:
    SEL_STYLE = {"negative": "danger", "neutral": "info", "positive": "success"}

    def __init__(self, session):
        self.s = session
        self.sent = None
        self.card, self.prog = W.HTML(), W.HTML()
        self.hint = W.HTML(HINT)
        self.sent_btn = {k: W.Button(description=SENT_DISPLAY[k], layout=W.Layout(width="150px", height="46px"))
                         for k in SENT_LABELS}
        for k, b in self.sent_btn.items():
            b.on_click(lambda _b, k=k: self._pick(k))
        self.ragu = W.Checkbox(value=False, description="Tandai ragu / ambigu", indent=False)
        self.btn_skip = W.Button(description="Lewati", icon="forward")
        self.btn_back = W.Button(description="Kembali", icon="backward")
        self.btn_skip.on_click(lambda _b: (self.s.skip(), self._render()))
        self.btn_back.on_click(lambda _b: (self.s.back(), self._render()))
        self.box = W.VBox([self.prog, self.card, self.hint, W.HBox(list(self.sent_btn.values())), self.ragu,
                           W.HBox([self.btn_back, self.btn_skip])])
        self._render()

    def _style(self):
        for k, b in self.sent_btn.items():
            b.button_style = self.SEL_STYLE[k] if k == self.sent else ""

    def _pick(self, k):                     # satu klik = simpan + lanjut
        if self.s.current() is None:
            return
        self.s.label(k, bool(self.ragu.value))
        self._render()

    def _render(self):
        done, total = self.s.done_count(), self.s.total()
        self.prog.value = (f"<b>Terlabel {done}/{total}</b> ({done / max(total, 1):.0%}) · "
                           f"antrean tersisa {len(self.s.queue)} · pelabel: <code>{html.escape(ANNOTATOR)}</code>")
        doc = self.s.current()
        if doc is None:
            self.card.value = "<h3>Selesai 🎉 Semua post dalam sampel sudah dilabeli.</h3>"
            self.sent, self.ragu.value = None, False
            self._style()
            return
        r = self.s.rows.loc[doc]
        dtype = html.escape(str(r["doc_type"])) if "doc_type" in r.index else "post"
        self.card.value = (
            "<div style='padding:12px 14px;border:1px solid rgba(128,128,128,.35);border-radius:8px;"
            "background:rgba(128,128,128,.08);max-width:760px;max-height:340px;overflow:auto'>"
            f"<div style='font-size:11px;opacity:.6;margin-bottom:6px'>telegram · {dtype} · {html.escape(doc)}</div>"
            f"<div style='font-size:16px;line-height:1.5'>{_esc(r['text_raw'])}</div></div>")
        old_sent, old_ragu = self.s.existing(doc)       # terisi saat membuka ulang lewat 'Kembali'
        self.sent = old_sent if old_sent in SENT_LABELS else None
        self.ragu.value = old_ragu
        self._style()

if HAVE_WIDGETS:
    session = LabelSession(tg, store)
    ui = LabelUI(session)
    display(ui.box)

## 5. Alternatif: labeling lewat Excel

1. `export_excel()` membuat `labeling_telegram_<ANNOTATOR>.xlsx` berisi post yang belum dilabeli, dengan **dropdown** pada kolom label.
2. Isi `label_sentiment` (opsional `ragu` dengan `x`), simpan.
3. `import_excel()` memasukkan label ke penyimpanan yang sama dengan UI. Baris kosong diabaikan, jadi boleh diisi sebagian.

File tidak ditimpa kalau sudah ada; pakai `overwrite=True` untuk membuat ulang.

In [15]:
def export_excel(overwrite=False):
    from openpyxl.worksheet.datavalidation import DataValidation
    path = OUT_DIR / f"labeling_telegram_{ANNOTATOR}.xlsx"
    if path.exists() and not overwrite:
        print(f"{path} sudah ada -> tidak ditimpa. Isi lalu import_excel(), atau export_excel(overwrite=True).")
        return path
    done = set(load_store()["doc_id"])
    todo = tg[~tg["doc_id"].isin(done)]
    out = pd.DataFrame({"doc_id": todo["doc_id"],
                        "doc_type": todo["doc_type"].astype(str) if "doc_type" in todo else "",
                        "teks": todo["text_raw"],
                        "label_sentiment": "", "ragu": ""})
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        out.to_excel(xw, index=False, sheet_name="label")
        ws = xw.sheets["label"]
        dv = DataValidation(type="list", formula1='"negative,neutral,positive"', allow_blank=True)
        ws.add_data_validation(dv)
        col = {n: i + 1 for i, n in enumerate(out.columns)}
        L = lambda name: ws.cell(row=1, column=col[name]).column_letter
        dv.add(f"{L('label_sentiment')}2:{L('label_sentiment')}{len(out) + 1}")
        for name, width in {"doc_id": 16, "doc_type": 11, "teks": 90, "label_sentiment": 16, "ragu": 7}.items():
            ws.column_dimensions[L(name)].width = width
        from openpyxl.styles import Alignment
        for row in ws.iter_rows(min_row=2, min_col=col["teks"], max_col=col["teks"]):
            for c in row:
                c.alignment = Alignment(wrap_text=True, vertical="top")
        ws.freeze_panes = "A2"
    print(f"{len(out)} post diekspor ke {path}")
    return path

def import_excel(path=None):
    path = Path(path) if path else OUT_DIR / f"labeling_telegram_{ANNOTATOR}.xlsx"
    x = pd.read_excel(path, sheet_name="label")
    x["label_sentiment"] = x["label_sentiment"].astype(str).str.strip().str.lower()
    ok = x[x["label_sentiment"].isin(SENT_LABELS)]
    st = load_store()
    for r in ok.itertuples():
        st = upsert(st, r.doc_id, r.label_sentiment, str(r.ragu).strip().lower() in ("x", "1", "ya", "true"))
    if len(ok):
        save_store(st)
    print(f"{len(ok)} label diimpor dari {len(x)} baris | total tersimpan: {len(st)}")
    return st

# contoh:  export_excel()   ... isi di Excel ...   import_excel()

## 6. Cek sebaran label

Sebaran Telegram hampir pasti **timpang ke Netral** (banyak jual-beli, tanya info, kehilangan barang). Itu wajar, tapi artinya: jangan andalkan akurasi, pakai **Macro F1** dan `class_weight="balanced"`. Kalau kelas Positif atau Negatif di bawah 10%, pertimbangkan menaikkan `N_SAMPLE`.

In [16]:
store = load_store()
lab = store.merge(tg[["doc_id"] + (["doc_type"] if "doc_type" in tg else [])], on="doc_id", how="left")
print(f"Terlabel: {len(lab)} / {len(tg)} | ditandai ragu: {int(lab['ragu'].sum()) if len(lab) else 0}\n")
if len(lab):
    share = lab["label_sentiment"].value_counts().reindex(SENT_LABELS).fillna(0)
    print("Jumlah :", share.astype(int).to_dict())
    print("Sebaran:", {k: f"{v:.0%}" for k, v in (share / share.sum()).items()})
    for k, v in (share / share.sum()).items():
        if v < 0.10:
            print(f"[!] Kelas '{k}' hanya {v:.0%} -> sedikit untuk pelatihan model; pertimbangkan menaikkan N_SAMPLE.")
    if "doc_type" in lab:
        display(pd.crosstab(lab["doc_type"].astype(str), lab["label_sentiment"]).reindex(columns=SENT_LABELS, fill_value=0))

Terlabel: 500 / 500 | ditandai ragu: 0

Jumlah : {'negative': 138, 'neutral': 314, 'positive': 48}
Sebaran: {'negative': '28%', 'neutral': '63%', 'positive': '10%'}
[!] Kelas 'positive' hanya 10% -> sedikit untuk pelatihan model; pertimbangkan menaikkan N_SAMPLE.


label_sentiment,negative,neutral,positive
doc_type,,,
post,138,314,48


## 7. Kesepakatan antar-pelabel & pembahasan selisih

Minta minimal satu orang lain melabeli **100-150 post pertama yang sama** (10-15% dari sampel; Pedoman menargetkan Cohen's κ di atas 0,6). Ia menjalankan notebook ini dengan `ANNOTATOR` berbeda dan **`SEED`, `N_SAMPLE`, `MIN_WORDS`, `DEDUP` yang sama**, lalu menaruh `labels_telegram_<nama>.parquet`-nya di `data/labeling/`. Pedoman: **κ ≥ 0,6** = substansial; di bawah 0,4 berarti panduan label perlu diperjelas.

In [17]:
files = sorted(OUT_DIR.glob("labels_telegram_*.parquet"))
stores = {f.stem.replace("labels_telegram_", ""): pd.read_parquet(f).set_index("doc_id") for f in files}
print("Pelabel ditemukan:", {k: len(v) for k, v in stores.items()})

if len(stores) < 2:
    print("\nBelum ada pelabel kedua -> lewati.")
else:
    names, rows = list(stores), []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = stores[names[i]], stores[names[j]]
            common = a.index.intersection(b.index)
            if len(common) < 30:
                continue
            ya, yb = a.loc[common, "label_sentiment"], b.loc[common, "label_sentiment"]
            rows.append({"pelabel_A": names[i], "pelabel_B": names[j], "n_overlap": len(common),
                         "persen_sama": round(float((ya == yb).mean()), 3),
                         "kappa": round(float(cohen_kappa_score(ya, yb)), 3)})
    display(pd.DataFrame(rows) if rows else "Overlap antar pelabel < 30 post.")

Pelabel ditemukan: {'annotator1': 500}

Belum ada pelabel kedua -> lewati.


In [ ]:
# Daftar selisih label + kasus ragu untuk dibahas bersama -> data/labeling/diskusi_telegram.xlsx
def build_discussion():
    if len(stores) < 1:
        print("Belum ada label."); return None
    names = list(stores)
    wide = pd.concat({n: s["label_sentiment"] for n, s in stores.items()}, axis=1)
    ragu = pd.concat({n: s["ragu"] for n, s in stores.items()}, axis=1).fillna(False).any(axis=1)
    wide["ada_ragu"] = ragu
    n_lab = wide[names].notna().sum(axis=1)
    differ = wide[names].nunique(axis=1, dropna=True) > 1
    sel = wide[(differ & (n_lab >= 2)) | wide["ada_ragu"]].copy()
    sel["alasan"] = np.where(differ.loc[sel.index], "beda label", "") 
    sel.loc[sel["ada_ragu"], "alasan"] = (sel.loc[sel["ada_ragu"], "alasan"] + " ragu").str.strip()
    sel = sel.reset_index().merge(tg[["doc_id", "text_raw"]], on="doc_id", how="left")
    sel["label_final"] = ""
    path = OUT_DIR / "diskusi_telegram.xlsx"
    sel.to_excel(path, index=False)
    print(f"{len(sel)} post untuk dibahas -> {path}")
    return sel

# build_discussion()   # jalankan setelah >= 2 pelabel selesai

## 8. Ekspor label Telegram

`MAIN_ANNOTATOR` adalah label acuan (setelah diskusi kasus ragu/selisih bila ada >1 pelabel). Jika kamu sudah mengisi kolom `label_final` di `diskusi_telegram.xlsx`, setel `DISCUSSION_PATH` agar label hasil diskusi menggantikan label pelabel utama.

Hasilnya `data/processed/telegram_labeled.parquet` (kolom `doc_id`, `label`, `ragu`, `annotator`). Di notebook GForm bagian 8, cukup isi:

```python
TELEGRAM_LABELS_PATH = ROOT / "data" / "processed" / "telegram_labeled.parquet"
```

In [18]:
MAIN_ANNOTATOR = ANNOTATOR
DISCUSSION_PATH = None     # contoh: OUT_DIR / "diskusi_telegram.xlsx"  (kolom label_final terisi)

m = pd.read_parquet(OUT_DIR / f"labels_telegram_{MAIN_ANNOTATOR}.parquet")[["doc_id", "label_sentiment", "ragu"]]
m = m.rename(columns={"label_sentiment": "label"})
m["annotator"] = MAIN_ANNOTATOR

if DISCUSSION_PATH:
    d = pd.read_excel(DISCUSSION_PATH)
    d["label_final"] = d["label_final"].astype(str).str.strip().str.lower()
    d = d[d["label_final"].isin(SENT_LABELS)].set_index("doc_id")["label_final"]
    m.loc[m["doc_id"].isin(d.index), "label"] = m.loc[m["doc_id"].isin(d.index), "doc_id"].map(d)
    m.loc[m["doc_id"].isin(d.index), "annotator"] = "konsensus"
    print(f"{len(d)} label diganti dengan hasil diskusi")

m = m[m["doc_id"].isin(tg["doc_id"])].drop_duplicates("doc_id")
out_path = ROOT / "data" / "processed" / "telegram_labeled.parquet"
out_path.parent.mkdir(parents=True, exist_ok=True)
m.to_parquet(out_path, index=False)
print(f"{len(m)} label Telegram -> {out_path}")
print(m["label"].value_counts().reindex(SENT_LABELS).fillna(0).astype(int).to_dict())

500 label Telegram -> c:\Users\yuhanidz\Nadi-Kampus-\data\processed\telegram_labeled.parquet
{'negative': 138, 'neutral': 314, 'positive': 48}


### Langkah berikutnya

1. Buka notebook GForm bagian 8, isi `TELEGRAM_LABELS_PATH` seperti di atas, jalankan → dapat `nadikampus_sentiment_labeled.parquet` dengan split train/val/test per sumber.
2. Bandingkan metode (lexicon InSet, model Indonesia siap pakai, pra-label LLM, TF-IDF + LogReg / IndoBERT) di **test set**, ukuran utama **Macro F1**, laporkan **Telegram dan GForm terpisah**.
3. Terapkan metode terbaik ke seluruh korpus; simpan `sentiment` dan `sentiment_prob`.

Catatan: label di sini adalah *gold set*, bukan label untuk seluruh korpus. Jangan latih lalu evaluasi pada post yang sama; pakai kolom `split`.